# Module 1: Teardown & Cleanup

## Overview

This notebook helps you clean up the resources created during Module 1. This is important to:
- Avoid ongoing AWS costs
- Clean up test environments
- Practice proper resource lifecycle management

### ⚠️ Warning

This will **destroy** your LangSmith deployment and associated AWS resources. Only run this if you're sure you want to remove everything.

**What will be destroyed:**
- Helm release (LangSmith application)
- Terraform-managed infrastructure (EKS, RDS, ElastiCache, S3, etc.)
- All associated data

**Estimated time:** 30-45 minutes


In [ ]:
# Bootstrap environment
import sys
from pathlib import Path

# Add notebooks directory to path so we can import shared as a package
# Find the notebooks directory by looking for the shared folder
possible_paths = [
    Path.cwd().parent,  # If cwd is module-1, go up one level to notebooks
    Path.cwd(),  # If cwd is already notebooks
    Path.cwd() / "notebooks",  # If cwd is workspace root
]

notebooks_path = None
for path in possible_paths:
    if path and (path / "shared" / "_bootstrap.py").exists():
        notebooks_path = path
        break

if not notebooks_path:
    notebooks_path = Path.cwd() / "notebooks"
    if not (notebooks_path / "shared" / "_bootstrap.py").exists():
        raise RuntimeError(f"Could not find notebooks/shared directory. Current dir: {Path.cwd()}")

# Add notebooks directory to path so 'shared' can be imported as a package
if str(notebooks_path) not in sys.path:
    sys.path.insert(0, str(notebooks_path))

from shared._bootstrap import bootstrap

# Run bootstrap
bootstrap_info = bootstrap()
artifacts_dir = Path(bootstrap_info['artifacts_dir'])
print(f"\nArtifacts directory: {artifacts_dir}")


## Confirmation

**⚠️ READ THIS CAREFULLY**

Before proceeding, confirm:
1. ✅ You want to destroy all resources
2. ✅ You've backed up any important data
3. ✅ You understand this cannot be undone
4. ✅ You're using the correct AWS account/region

**Double-check your AWS account and region before proceeding!**


In [ ]:
import os
from shared._validation import require_env
from shared._aws_helpers import aws_region, sts_identity

# Show current AWS session
config = require_env("CLUSTER_NAME", "AWS_REGION", "NAMESPACE", "HELM_RELEASE")
region = aws_region()
identity = sts_identity()

print("### Current AWS Session")
print("=" * 60)
print(f"Account ID: {identity['Account']}")
print(f"Region: {region}")
print(f"User ARN: {identity['Arn']}")
print("=" * 60)

print(f"\n### Resources to be Destroyed")
print(f"Cluster: {config['CLUSTER_NAME']}")
print(f"Namespace: {config['NAMESPACE']}")
print(f"Helm Release: {config['HELM_RELEASE']}")
print("=" * 60)

print("\n⚠️  VERIFY THE ABOVE INFORMATION IS CORRECT!")
print("💡 If this is the wrong account/region, STOP NOW and update your .env file")


## Step 1: Uninstall Helm Release

First, we'll uninstall the LangSmith Helm release. This removes the application but leaves the infrastructure.


In [ ]:
from shared._shell import run
from shared._aws_helpers import aws_region

cluster_name = config["CLUSTER_NAME"]
namespace = config["NAMESPACE"]
helm_release = config["HELM_RELEASE"]
region = aws_region()

# Ensure kubectl is configured
print("### Configuring kubectl\n")
run(
    ["aws", "eks", "update-kubeconfig", "--name", cluster_name, "--region", region],
    check=True,
    stream=False
)

# Check if Helm release exists
print(f"\n### Checking Helm Release: {helm_release}\n")
result = run(
    ["helm", "list", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

import json
releases = json.loads(result.stdout) if result.returncode == 0 else []
langsmith_releases = [r for r in releases if r.get("name") == helm_release]

if langsmith_releases:
    release = langsmith_releases[0]
    print(f"Found Helm release: {release['name']}")
    print(f"Status: {release['status']}")
    print(f"Chart: {release['chart']}")
    
    print(f"\n⚠️  UNCOMMENT THE CODE BELOW TO UNINSTALL HELM RELEASE")
    print("This will remove the LangSmith application from the cluster.\n")
    
    # UNCOMMENT TO UNINSTALL:
    # print("Uninstalling Helm release...\n")
    # result = run(
    #     ["helm", "uninstall", helm_release, "-n", namespace],
    #     check=True,
    #     stream=True
    # )
    # print("\n✅ Helm release uninstalled")
else:
    print(f"Helm release '{helm_release}' not found")
    print("💡 It may have already been uninstalled, or the namespace is different")


## Step 2: Clean Up Kubernetes Resources

Remove any remaining Kubernetes resources (secrets, PVCs, etc.) that might not be cleaned up by Helm.


In [ ]:
# Check for remaining resources
print("### Checking for Remaining Kubernetes Resources\n")

# List resources in namespace
resources_to_check = [
    ("pods", ["kubectl", "get", "pods", "-n", namespace]),
    ("services", ["kubectl", "get", "svc", "-n", namespace]),
    ("secrets", ["kubectl", "get", "secrets", "-n", namespace]),
    ("pvc", ["kubectl", "get", "pvc", "-n", namespace]),
]

remaining = []
for resource_type, cmd in resources_to_check:
    result = run(cmd, check=False, stream=False)
    if result.returncode == 0:
        lines = result.stdout.strip().split('\n')
        # Skip header line
        if len(lines) > 1:
            remaining.append(resource_type)
            print(f"⚠️  Found {len(lines) - 1} {resource_type}(s)")

if remaining:
    print(f"\n💡 The following resource types still exist: {', '.join(remaining)}")
    print("   You may want to clean these up manually:")
    print(f"   kubectl delete all --all -n {namespace}")
    print(f"   kubectl delete pvc --all -n {namespace}")
    print(f"   kubectl delete secrets --all -n {namespace}")
else:
    print("✅ No remaining resources found (or namespace is empty)")


## Step 3: Destroy Terraform Infrastructure

**⚠️ CRITICAL:** This will destroy all AWS infrastructure including:
- EKS cluster
- RDS PostgreSQL database (and all data)
- ElastiCache Redis (and all data)
- S3 buckets (and all data)
- IAM roles and policies
- VPC resources (if managed by Terraform)

**This cannot be undone!**


In [ ]:
from pathlib import Path

terraform_dir = Path(config.get("TERRAFORM_DIR", "")).expanduser().resolve()

if not terraform_dir.exists():
    print(f"⚠️  Terraform directory not found: {terraform_dir}")
    print("💡 Update TERRAFORM_DIR in your .env file, or destroy infrastructure manually")
else:
    print(f"### Terraform Directory: {terraform_dir}\n")
    
    # Check Terraform state
    print("Checking Terraform state...\n")
    result = run(
        ["terraform", "show", "-json"],
        cwd=str(terraform_dir),
        check=False,
        stream=False
    )
    
    if result.returncode == 0:
        state_data = json.loads(result.stdout)
        if state_data.get("values") and state_data["values"].get("root_module"):
            resources = state_data["values"]["root_module"].get("resources", [])
            print(f"Found {len(resources)} resources in Terraform state")
            print("⚠️  These will all be destroyed!\n")
        else:
            print("Terraform state appears empty or not initialized")
    else:
        print("Could not read Terraform state")
        print("💡 Terraform may not be initialized, or state file doesn't exist")
    
    print("⚠️  UNCOMMENT THE CODE BELOW TO DESTROY TERRAFORM INFRASTRUCTURE")
    print("This will destroy ALL resources managed by Terraform.\n")
    
    # UNCOMMENT TO DESTROY:
    # print("Destroying Terraform infrastructure...")
    # print("This will take 15-30 minutes...\n")
    # 
    # result = run(
    #     ["terraform", "destroy", "-auto-approve"],
    #     cwd=str(terraform_dir),
    #     check=False,  # Don't fail on errors, we'll check return code
    #     stream=True
    # )
    # 
    # # Save destroy output
    # destroy_file = artifacts_dir / "terraform-destroy.txt"
    # with open(destroy_file, "w") as f:
    #     f.write(result.stdout)
    #     if result.stderr:
    #         f.write("\n\nSTDERR:\n")
    #         f.write(result.stderr)
    # 
    # if result.returncode == 0:
    #     print("\n✅ Terraform destroy completed successfully")
    #     print(f"💡 Destroy output saved to: {destroy_file}")
    # else:
    #     print(f"\n⚠️  Terraform destroy had issues (rc={result.returncode})")
    #     print("💡 Review the output above for errors")
    #     print(f"   Destroy output saved to: {destroy_file}")
    
    print("💡 To destroy, edit this cell and uncomment the code above")


## Step 4: Verify Cleanup

After teardown, verify that resources have been removed.


In [ ]:
from shared._aws_helpers import eks_cluster_exists

print("### Verifying Cleanup\n")

# Check if cluster still exists
cluster_name = config["CLUSTER_NAME"]
region = aws_region()

if eks_cluster_exists(cluster_name):
    warn(f"Cluster '{cluster_name}' still exists")
    print("💡 Terraform destroy may not have completed, or cluster was created outside Terraform")
else:
    ok(f"Cluster '{cluster_name}' does not exist (destroyed or never created)")

# Check for remaining S3 buckets (if we know the bucket name)
print("\n### S3 Buckets\n")
print("💡 Check AWS console for any remaining S3 buckets")
print("   Terraform should have destroyed buckets it created, but verify manually")

# Check for remaining RDS instances
print("\n### RDS Instances\n")
print("💡 Check AWS console for any remaining RDS instances")
print("   Terraform should have destroyed RDS instances it created")

# Check for remaining ElastiCache clusters
print("\n### ElastiCache Clusters\n")
print("💡 Check AWS console for any remaining ElastiCache clusters")
print("   Terraform should have destroyed ElastiCache clusters it created")

print("\n✅ Cleanup verification complete")
print("💡 Review AWS console to ensure all resources are removed")


## Summary

### ✅ Teardown Checklist

- [ ] Helm release uninstalled
- [ ] Kubernetes resources cleaned up
- [ ] Terraform infrastructure destroyed
- [ ] EKS cluster removed
- [ ] RDS instance removed
- [ ] ElastiCache cluster removed
- [ ] S3 buckets removed (or emptied)
- [ ] AWS console verified (no remaining resources)

### 💡 Important Notes

1. **Data Loss:** All data in RDS, ElastiCache, and S3 has been permanently deleted
2. **Costs:** You should see AWS costs stop accruing within 24 hours
3. **Artifacts:** Diagnostic artifacts in `artifacts/` directory are preserved for reference
4. **Re-deployment:** You can re-run Module 1 notebooks to create a fresh deployment

### 🎯 Next Steps

If you want to start over:
1. Review and update your `.env` file
2. Run `01_aws_preflight.ipynb` again
3. Proceed through the module notebooks

**Thank you for completing Module 1!**
